# Graph RAG (Retrieval Augmented Generation)

## Task Description
Perform Graph RAG by extracting a Knowledge Graph from text and using it to answer questions.

In this notebook, we will:
1. Install necessary libraries (`langchain`, `networkx`, `transformers`).
2. Import modules.
3. Load a sample text document.
4. Clean and prepare the text.
5. Load `Qwen/Qwen2.5-1.5B-Instruct` for entity extraction and reasoning.
6. Inference: Extract Knowledge Graph, Visualize it, and Query it.
7. Save Results.
8. Advanced Usage.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## 1. Installation
Install `networkx`, `matplotlib`, `transformers`, `accelerate`, `bitsandbytes`.

In [ ]:
!pip install -q networkx matplotlib transformers torch accelerate bitsandbytes

## 2. Imports
Import necessary libraries.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import re

## 3. Load Data
We will use a sample text about 'The History of AI' to build our graph.

In [ ]:
text_data = """
Artificial Intelligence was founded as an academic discipline in 1956. 
Alan Turing published 'Computing Machinery and Intelligence' in 1950. 
The Dartmouth Workshop was organized by John McCarthy. 
Deep Learning revolutionized AI in the 2010s. 
Neural Networks are a subset of Machine Learning. 
Transformers were introduced by Google researchers in 2017.
"""
print("Text loaded.")

## 4. Data Cleaning
Split text into sentences.

In [ ]:
sentences = [s.strip() for s in text_data.split('.') if s.strip()]
print(f"Processing {len(sentences)} sentences.")

## 5. Load Model
Load `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=128)
print("Model loaded.")

## 6. Inference
1. Extract Triplets (Subject, Relation, Object).
2. Build Graph.
3. Visualize.
4. Query.

In [ ]:
def extract_triplets(text):
    prompt = f"Extract triplets (Subject | Relation | Object) from this text: '{text}'. Return only the triplets in the format 'Subject | Relation | Object'. Do not explain."
    messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts Knowledge Graph triplets."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(text)
    response = outputs[0]["generated_text"].split("<|im_start|>assistant")[-1].strip()
    triplets = []
    for line in response.split('\n'):
        if '|' in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) == 3:
                triplets.append(parts)
    return triplets

# Build Graph
G = nx.DiGraph()
all_triplets = []
for sent in sentences:
    triplets = extract_triplets(sent)
    all_triplets.extend(triplets)
    for sub, rel, obj in triplets:
        G.add_edge(sub, obj, relation=rel)

# Visualize
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000, font_size=10, font_weight='bold')
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.title("Extracted Knowledge Graph")
plt.savefig("graph_visualization.png")
plt.show()

# Graph RAG Query
query_entity = "Alan Turing"
if query_entity in G:
    neighbors = list(G.neighbors(query_entity)) + list(G.predecessors(query_entity))
    context = f"Neighbors of {query_entity}: {neighbors}. Relations: {G.edges(query_entity, data=True)}"
    print(f"Graph Context: {context}")
else:
    print(f"Entity {query_entity} not found in graph.")

## 7. Save Results
Save the extracted triplets and the graph image.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage
Use a vector database (like FAISS or Pinecone) to index nodes for fuzzy retrieval, or use `neo4j` for persistent storage.

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print("Try integrating with LangChain GraphStore!")